# 05 · Feature groups: what do the clinical features contribute on their own?

Same 5-fold patient-level CV on 80% of Hospital A and same weighted logistic regression as notebook 03. Only the set of input columns changes:

| feature set | inputs | question it answers |
|---|---|---|
| all features | vitals + labs + demographics and time | the reference |
| clinical only | vitals + labs | what does the physiology alone achieve? |
| demographics + time only | Age, Gender, Unit1, Unit2, HospAdmTime, ICULOS | what can the model do *without* any physiology? |
| all except ICULOS | everything but the hours-in-ICU counter | how much rests on the ICU clock? |

No missingness features here, so the comparison is about the raw feature groups only.

In [1]:
import sepsis as s

data_A = s.load_hospital("A")
dev_ids, test_ids = s.split_patients(data_A, test_size=0.20)      # same 80/20 split and folds as before
fold_of = s.make_folds(data_A, dev_ids)

# No flags / hours-since columns needed in this notebook, so they are not built (saves memory)
dev = s.prepare_dev(data_A, dev_ids, fold_of, missingness=False)

In [2]:
FEATURE_SETS = {
    "all features": s.ALL_FEATURES,                                   # the reference: compared against below
    "clinical only": s.CLINICAL,
    "demographics + time only": s.DEMOGRAPHICS,
    "all except ICULOS": [c for c in s.ALL_FEATURES if c != "ICULOS"],
}
oofs = {name: s.run_cv(dev, cols) for name, cols in FEATURE_SETS.items()}

trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 36 input features
trained on 30 input features
trained on 30 input features
trained on 30 input features
trained on 30 input features
trained on 30 input features
trained on 6 input features
trained on 6 input features
trained on 6 input features
trained on 6 input features
trained on 6 input features
trained on 35 input features
trained on 35 input features
trained on 35 input features
trained on 35 input features
trained on 35 input features


## 1. AUROC and AUPRC

The second table is each set minus *all features*, fold by fold. Negative = worse than using everything.

In [3]:
summary, diffs = s.compare_auc(oofs)
display(summary.round(4))
display(diffs.round(4))

AUROC           AUPRC        
                            mean     std    mean     std
Features                                                
all features              0.7462  0.0230  0.0827  0.0142
clinical only             0.6739  0.0157  0.0461  0.0051
demographics + time only  0.6987  0.0177  0.0765  0.0139
all except ICULOS         0.6951  0.0175  0.0488  0.0065

AUROC                                             \
Features clinical only demographics + time only all except ICULOS   
1              -0.0790                  -0.0568           -0.0570   
2              -0.0645                  -0.0353           -0.0455   
3              -0.0603                  -0.0481           -0.0389   
4              -0.0863                  -0.0450           -0.0725   
5              -0.0717                  -0.0526           -0.0417   
mean           -0.0724                  -0.0476           -0.0511   
std             0.0106                   0.0082            0.0138   

                 AUPRC                                             
Features clinical only demographics + time only all except ICULOS  
1              -0.0498                  -0.0118           -0.0438  
2              -0.0335                  -0.0075           -0.0312  
3              -0.0255                  -0.0068           -0.0236  
4              -0.0446                   0.0005           -0.0439  
5              -0.0298                  -0.0054           -0.0270  
mean           -0.0366                  -0.0062           -0.0339  
std             0.0102                   0.0044            0.0095

## 2. Each feature set at its own best cutoff

In [4]:
s.best_cutoff_table(oofs).round(4)

,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
all features,0.55,0.1094,0.0608,0.5454,0.1871,0.3099,0.3098,0.0379
clinical only,0.55,0.0797,0.0435,0.4686,0.2286,0.1740,0.1739,0.0201
demographics + time only,0.60,0.1456,0.0904,0.3738,0.0835,0.2737,0.2736,0.0252
all except ICULOS,0.55,0.0829,0.0451,0.5059,0.2377,0.1987,0.1987,0.0320


## Conclusion

All features together are clearly best (AUROC about 0.746, AUPRC 0.083, best utility 0.31). The surprise is the next two rows: **demographics + time alone** (six columns, no physiology at all) reach an AUROC of about 0.699 and a utility of 0.27, while **vitals and labs alone** only reach 0.674 and 0.17. Removing just ICULOS costs about 0.05 AUROC and drops AUPRC to 0.049, almost as much as removing all six demographic columns.

So a large part of what the model knows comes from *how long the patient has been in the ICU*, not from their physiology. That can be a real pattern (the longer the stay, the more chances to develop sepsis), but it can also be a shortcut that will not carry over to another hospital. Notebook 06 looks at this directly. The 20% test set and Hospital B are kept untouched.